# 03 심화: 30기 Walker-Delta 재방문 toy simulation

원형 2체 궤도, 지구 자전, nadir-pointing sensor를 사용해 5면·6면과 모든 phase를 비교한다. 북한 경계의 작은 대표 grid만 쓰므로 논문 수치 재현이 아니다.

In [1]:
from math import sin,cos,sqrt,pi,acos,radians
from statistics import mean

MU=398600.4418; R=6378.137; A=R+500
N=sqrt(MU/A**3); OMEGA_E=7.2921159e-5
INC=radians(80); HALF_FOV=radians(15)  # 논문의 FOV 30°를 full cone으로 가정
targets=[(lat,lon) for lat in (38.5,39.5,40.5,41.5,42.0)
                   for lon in (125.0,126.5,128.0,129.5,130.5)]
len(targets), 2*pi/N/60

(25, 94.61630047543098)

In [2]:
def dot(a,b): return sum(x*y for x,y in zip(a,b))
def norm(a): return sqrt(dot(a,a))
def sat_eci(raan,u):
    # 원궤도 좌표를 R3(RAAN) R1(inclination)으로 회전
    return (A*(cos(raan)*cos(u)-sin(raan)*sin(u)*cos(INC)),
            A*(sin(raan)*cos(u)+cos(raan)*sin(u)*cos(INC)),
            A*sin(u)*sin(INC))
def ground_eci(lat_deg,lon_deg,t):
    lat=radians(lat_deg); theta=radians(lon_deg)+OMEGA_E*t
    return (R*cos(lat)*cos(theta),R*cos(lat)*sin(theta),R*sin(lat))
def visible(s,g):
    # horizon 조건: 지상점에서 위성이 지평선 위에 있어야 한다.
    if dot(s,g) <= R*R: return False
    los=tuple(gi-si for si,gi in zip(s,g)); nadir=tuple(-si for si in s)
    c=max(-1,min(1,dot(los,nadir)/(norm(los)*norm(nadir))))
    return acos(c) <= HALF_FOV

assert visible((A,0,0),(R,0,0))
assert not visible((A,0,0),(-R,0,0))

In [3]:
def constellation(planes,phase):
    per_plane=30//planes; states=[]
    for j in range(planes):
        raan=2*pi*j/planes
        for k in range(per_plane):
            u0=2*pi*k/per_plane + 2*pi*phase*j/(per_plane*planes)
            states.append((raan,u0))
    return states

def event_intervals(times,flags):
    out=[]; start=None; prev=None
    for t,on in zip(times,flags):
        if on and start is None: start=t
        if not on and start is not None: out.append((start,prev)); start=None
        prev=t
    if start is not None: out.append((start,prev))
    return out

def simulate(planes,phase,duration_s=86400,step_s=120):
    times=list(range(0,duration_s+1,step_s)); sats=constellation(planes,phase)
    all_gaps=[]; observed_targets=0
    for lat,lon in targets:
        flags=[]
        for t in times:
            g=ground_eci(lat,lon,t)
            flags.append(any(visible(sat_eci(raan,u0+N*t),g) for raan,u0 in sats))
        events=event_intervals(times,flags)
        gaps=[b[0]-a[1] for a,b in zip(events,events[1:])]
        if events: observed_targets+=1
        all_gaps.extend(gaps)
    return {'planes':planes,'phase':phase,'observed_targets':observed_targets,
            'gap_count':len(all_gaps),
            'mean_min':mean(all_gaps)/60 if all_gaps else None,
            'max_min':max(all_gaps)/60 if all_gaps else None}


In [4]:
results=[simulate(p,f) for p in (5,6) for f in range(p)]
for r in results:
    print(r)

{'planes': 5, 'phase': 0, 'observed_targets': 22, 'gap_count': 33, 'mean_min': 246.66666666666666, 'max_min': 940.0}
{'planes': 5, 'phase': 1, 'observed_targets': 23, 'gap_count': 27, 'mean_min': 357.18518518518516, 'max_min': 936.0}
{'planes': 5, 'phase': 2, 'observed_targets': 25, 'gap_count': 28, 'mean_min': 523.5, 'max_min': 864.0}
{'planes': 5, 'phase': 3, 'observed_targets': 16, 'gap_count': 12, 'mean_min': 343.5, 'max_min': 932.0}
{'planes': 5, 'phase': 4, 'observed_targets': 19, 'gap_count': 39, 'mean_min': 450.56410256410254, 'max_min': 574.0}
{'planes': 6, 'phase': 0, 'observed_targets': 16, 'gap_count': 29, 'mean_min': 370.2068965517241, 'max_min': 1192.0}
{'planes': 6, 'phase': 1, 'observed_targets': 25, 'gap_count': 26, 'mean_min': 469.61538461538464, 'max_min': 952.0}
{'planes': 6, 'phase': 2, 'observed_targets': 19, 'gap_count': 31, 'mean_min': 234.1290322580645, 'max_min': 788.0}
{'planes': 6, 'phase': 3, 'observed_targets': 25, 'gap_count': 31, 'mean_min': 517.41935483

toy 결과가 논문의 43~47분과 달라도 실패가 아니다. grid, FOV 의미, epoch, event interpolation과 지구모델이 다르기 때문이다. 중요한 것은 같은 코드와 가정으로 5면·6면·phase를 공정하게 비교하는 것이다.

In [5]:
# time step 민감도: coarse sampling은 event 경계와 짧은 pass를 왜곡한다.
sensitivity=[]
for step in (60,120,300):
    r=simulate(6,1,duration_s=86400,step_s=step)
    sensitivity.append((step,r['gap_count'],round(r['mean_min'],2),round(r['max_min'],2)))
sensitivity

[(60, 76, 284.83, 952.0), (120, 26, 469.62, 952.0), (300, 5, 868.0, 1195.0)]

In [6]:
assert all(r['observed_targets']>0 and r['gap_count']>0 for r in results)
assert all(r['mean_min']<=r['max_min'] for r in results)
print('모든 구성에서 event와 gap이 생성되었고 통계 불변조건을 만족했습니다.')

모든 구성에서 event와 gap이 생성되었고 통계 불변조건을 만족했습니다.


## 고급 확장

1. 북한 polygon과 0.1° grid를 사용하고 면적 가중치를 적용한다.
2. FOV를 full-angle/half-angle 두 경우로 나눠 sensitivity를 보고한다.
3. J2에 의한 RAAN drift, drag, 발사 분산과 station-keeping을 추가한다.
4. 태양 고도·구름·slew·저장공간·downlink를 넣어 유효 영상 delivery revisit를 계산한다.